In [ ]:
code = 'B120_RE_PSL'
pickle_path = 'C:/PICKLE/'
parameter_path = f'Parameter_{code}.csv'
meta_data_path = f"Parameter_{code}_MetaData.csv"
output_csv_path = f'{code}_output/'

from pgcbacktest.BtParameters import *
from pgcbacktest.BacktestOptions import *

try:
    parameter, parameter_len = get_parameter_data(code, parameter_path)
    meta_data, meta_row_nos = get_meta_data(code, meta_data_path)
    os.makedirs(output_csv_path, exist_ok=True)
except Exception as e:
    input(str(e))

In [ ]:
def b120_per_minute_mtm(bt, start_time, end_time, orderside, method, sl, ut_sl, om):
    try:
        start_dt = datetime.datetime.combine(bt.current_date, start_time)
        end_dt = datetime.datetime.combine(bt.current_date, end_time)
        end_dt_1m = end_dt + datetime.timedelta(minutes=10)

        ce_scrip, pe_scrip, ce_price, pe_price, future_price, start_dt = bt.get_strike(start_dt, end_dt, om=om)
        if ce_scrip is None: return '', None

        from_candle_close = True if method == 'CC' else False

        entry_time = start_dt
        _, _, _, _, ce_sl_price, ce_sl_time, ce_mtm_data = bt.sl_check_single_leg(start_dt, end_dt, ce_scrip, sl=sl, with_ohlc=True, orderside=orderside, from_candle_close=from_candle_close, per_minute_mtm=True)
        _, _, _, _, pe_sl_price, pe_sl_time, pe_mtm_data = bt.sl_check_single_leg(start_dt, end_dt, pe_scrip, sl=sl, with_ohlc=True, orderside=orderside, from_candle_close=from_candle_close, per_minute_mtm=True)
        ce_sl_time = ce_sl_time if ce_sl_time else end_dt_1m
        pe_sl_time = pe_sl_time if pe_sl_time else end_dt_1m

        ut_sl = ut_sl if str(ut_sl) == 'TTC' else float(ut_sl)
        
        if ce_sl_time < pe_sl_time:
            pe_sl_time = end_dt_1m
            
            ut = 'PE'
            pe_mtm_data = pe_mtm_data[pe_mtm_data.index <= ce_sl_time]
            
            ut_sl_price = pe_price if str(ut_sl) == 'TTC' else None
            ut_open, _, _, _, _, ut_sl_time, ut_mtm_data = bt.sl_check_single_leg(ce_sl_time, end_dt, pe_scrip, sl=ut_sl, sl_price=ut_sl_price, with_ohlc=True, pl_with_slipage=False, orderside=orderside, from_candle_close=from_candle_close, per_minute_mtm=True)

            if ut_open:
                if (str(ut_sl) == 'TTC') and (ut_open > ut_sl_price):
                    ut_sl_price = pe_sl_price
                    _, _, _, _, _, ut_sl_time, ut_mtm_data = bt.sl_check_single_leg(ce_sl_time, end_dt, pe_scrip, sl=ut_sl, sl_price=ut_sl_price, with_ohlc=True, pl_with_slipage=False, orderside=orderside, from_candle_close=from_candle_close, per_minute_mtm=True)

        elif pe_sl_time < ce_sl_time:
            ce_sl_time = end_dt_1m
            
            ut = 'CE'
            ce_mtm_data = ce_mtm_data[ce_mtm_data.index <= pe_sl_time]
            
            ut_sl_price = ce_price if str(ut_sl) == 'TTC' else None
            ut_open, _, _, _, _, ut_sl_time, ut_mtm_data = bt.sl_check_single_leg(pe_sl_time, end_dt, ce_scrip, sl=ut_sl, sl_price=ut_sl_price, with_ohlc=True, pl_with_slipage=False, orderside=orderside, from_candle_close=from_candle_close, per_minute_mtm=True)

            if ut_open:
                if (str(ut_sl) == 'TTC') and (ut_open > ut_sl_price):
                    ut_sl_price = ce_sl_price
                    _, _, _, _, _, ut_sl_time, ut_mtm_data = bt.sl_check_single_leg(pe_sl_time, end_dt, ce_scrip, sl=ut_sl, sl_price=ut_sl_price, with_ohlc=True, pl_with_slipage=False, orderside=orderside, from_candle_close=from_candle_close, per_minute_mtm=True)
        else:
            ut, ut_open = '', ''
            ut_sl_time, ut_mtm_data = '', pd.Series()
            
        if ut_open:
            b120_sl_time = ut_sl_time
        else:
            b120_sl_time = max(ce_sl_time, pe_sl_time)

        b120_sl_time = '' if b120_sl_time == end_dt_1m else b120_sl_time

        ce_mtm_data = set_pm_time_index(ce_mtm_data, time_index)
        pe_mtm_data = set_pm_time_index(pe_mtm_data, time_index)
        
        if ut:
            ut_mtm_data = set_pm_time_index(ut_mtm_data, time_index)
            return b120_sl_time, {'STD':ce_mtm_data+pe_mtm_data, 'UT':ut_mtm_data}
        else:
            return b120_sl_time, {'STD':ce_mtm_data+pe_mtm_data}
    
    except Exception as e:
        print(e, [bt.index, bt.current_date, start_time, end_time, orderside, method, sl, ut_sl, om])
        return '', None

In [ ]:
def b120_RE_per_minute_mtm(bt, start_time, end_time, orderside, method, sl, ut_sl, om):
    try:
        end_dt = datetime.datetime.combine(bt.current_date, end_time)
        
        b120_sl_time, b120_0_mtm_data = b120_per_minute_mtm(bt, start_time, end_time, orderside, method, sl, ut_sl, om)
        time_index = get_pm_time_index(bt.current_date, bt.meta_start_time, bt.meta_end_time)

        rtn = {}
        rtn['STD0'] = b120_0_mtm_data['STD']
        if 'UT' in b120_0_mtm_data: rtn['UT0'] = b120_0_mtm_data['UT']
        
        for re_no in range(max_re):
            if b120_sl_time and re_no < re_entries and b120_sl_time < end_dt - datetime.timedelta(minutes=5):
                start_time = b120_sl_time.time()
                b120_sl_time, b120_re_mtm_data = b120_per_minute_mtm(bt, start_time, end_time, orderside, method, sl, ut_sl, om)
                if b120_re_mtm_data is not None:
                    rtn[f'STD{re_no+1}'] = b120_re_mtm_data['STD']
                    if 'UT' in b120_re_mtm_data: rtn[f'UT{re_no+1}'] = b120_re_mtm_data['UT']
            else:
                break

        return rtn
    except Exception as e:
        print(e, [bt.index, bt.current_date, start_time, end_time, orderside, method, sl, ut_sl, om])
        return

In [ ]:
def b120_RE_PSL(bt, start_time, end_time, last_trade_time, trade_interval, orderside, method, sl, ut_sl, om):
    try:
        start_dt = datetime.datetime.combine(bt.current_date, start_time)
        end_dt = datetime.datetime.combine(bt.current_date, end_time)
        last_trade_dt = datetime.datetime.combine(bt.current_date, last_trade_time)

        entry_time = start_dt
        time_range = pd.date_range(start_dt, last_trade_dt, freq=trade_interval.lower()).time
        
        per_minute_trades = {re_time: b120_RE_per_minute_mtm(bt, re_time, end_time, orderside, method, sl, ut_sl, om) for re_time in time_range}
        tkeys = sorted(set(k for v in per_minute_trades.values() for k in v.keys()))
        per_minute_trades = {k: v for k, v in per_minute_trades.items() if v is not None}
        per_minute_trades = {k: (np.sum([v[k] for v in per_minute_trades.values() if k in v], axis=0) if k in tkeys else np.array(empty_time_series)) for k in all_keys}
        
        if only_mtm:
            mtm_time_lists = {k: list(v) + [v.iloc[-1]] for k, v in per_minute_trades.items()}
        else:
            per_minute_mtm = np.sum(list(per_minute_trades.values()), axis=0)
            
            total_minutes = len(time_range)
            per_minute_margin = int(fund/total_minutes)
            shares_per_minute = int(per_minute_margin/margin_per_share)
            per_minute_mtm_total = per_minute_mtm * shares_per_minute

            mtm_dict_idx = {}
            for mtm_percent, threshold in check_mtms.items():
                condition = np.where(per_minute_mtm_total > threshold)[0] if threshold > 0 else np.where(per_minute_mtm_total < threshold)[0]

                if condition.size > 0:
                    idx = int(condition[0])
                    mtm_dict_idx[mtm_percent] = (time_index[idx], idx + 1)
                else:
                    mtm_dict_idx[mtm_percent] = ('', -1)
                
            mtm_time_lists = {}
            for key in all_keys:
                mtm_time_lists[key] = [fund] + [per_minute_trades[key][item]*shares_per_minute if isinstance(item, int) else item for value in mtm_dict_idx.values() for item in value]
           
        rtn = []
        for key in all_keys:
            rtn.append([code, bt.index, start_time, end_time, last_trade_time, trade_interval, orderside, method, sl, cv(ut_sl), om, key, bt.current_date.date(), bt.current_date.day_name(), bt.dte, entry_time.time(), future_price] + mtm_time_lists[key])
        
        return rtn
    except Exception as e:
        print(e, [bt.index, bt.current_date, start_time, end_time, last_trade_time, trade_interval, orderside, method, sl, ut_sl, om])
        return

In [ ]:
for row_idx in range(len(meta_data)):

    if row_idx in meta_row_nos and meta_data.loc[row_idx, 'run']:
        try:
            meta_row = meta_data.iloc[row_idx]
            index, dte, from_date, to_date, start_time, end_time, date_lists = get_meta_row_data(meta_row, pickle_path)
            re_entries = int(parameter['re_entries'].max())
            max_re = max(7, re_entries)
            
            log_cols = ('P_Strategy/P_Index/P_StartTime/P_EndTime/P_LastTradeTime/P_TradeInterval/P_OrderSide/P_Method/P_SL/P_UTSL/P_OM/MtmKey/Date/Day/DTE/Entry.Time/Future/')
            
            all_keys = []
            for re in range(re_entries+1):
                all_keys.extend([f'STD{re}', f'UT{re}'])
            
            only_mtm = False
            if only_mtm:
                log_time_col = get_pm_time_index(datetime.datetime.now(), start_time, end_time).time
                log_cols += '/'.join(map(str, log_time_col))
                log_cols += '/Final.PNL'
            else:
                notinal_value = 12
                fund = 100_00_00_00
                mtm_percent_stop = [-0.1, -0.2, -0.3, -0.4, -0.5, -0.6, -0.7, -0.8, -0.9, -1, -1.25, -1.5, -1.75, -2, -2.5, -3, -3.5, -4, -100, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1, 1.25, 1.5, 1.75, 2, 2.5, 3, 3.5, 4]
                log_cols += 'Fund'
                for mtmp in mtm_percent_stop:
                    log_cols += f'/{mtmp:.2f}.Time/{mtmp:.2f}.PNL'

            log_cols = log_cols.split('/')

            for current_date in date_lists:

                file_name = f"{index} {current_date.date()} {code}"
                if not is_file_exists(output_csv_path, file_name, parameter_len):

                    t1 = datetime.datetime.now()
                    print(f"Row-{row_idx} | File-{file_name} | Total-{parameter_len}")
                    
                    bt = IntradayBacktest(pickle_path, index, current_date, dte, start_time, end_time)
                    time_index = get_pm_time_index(bt.current_date, bt.meta_start_time, bt.meta_end_time)
                    empty_time_series = set_pm_time_index(pd.Series(), time_index)
                    future_price = bt.future_data['close'].iloc[0]
                    
                    if not only_mtm:
                        margin_per_share = future_price * (notinal_value / 100)
                        check_mtms = {mtm_percent: fund * mtm_percent / 100 for mtm_percent in mtm_percent_stop}
                    
                    for idx, i in enumerate(range(0, parameter_len, chunk_size), start=1):
                        chunck_file_name = f"{output_csv_path}{file_name} No-{idx}.parquet"
                        print(chunck_file_name)
                        
                        chunk_parameter = parameter.iloc[i:i+chunk_size]
                        chunk = [b120_RE_PSL(bt, row.entry_time, row.exit_time, row.last_trade_time, row.trade_interval, row.orderside, row.method, row.sl, row.ut_sl, row.om) for row in tqdm(chunk_parameter.itertuples(), total=len(chunk_parameter), colour='GREEN')]
                        chunk = [sublist for group in chunk if group is not None for sublist in group]
                        save_chunk_data(chunk, log_cols, chunck_file_name)
                        
                        del chunk
                        del chunk_parameter
                        gc.collect()

                    del bt
                    gc.collect()
                    
                    t2 = datetime.datetime.now()
                    print(t2-t1)
        except Exception as e:
            input(str(e))